# Clean DEM, divide elevation bins for each basin

Builds a coarsened DEM for the study basins and splits each basin into elevation bins.

**Steps**
1. Merge the ASTER GDEM tiles and coarsen from 1-arcsec to 15-arcsec (~450 m) pixels.
2. Clip to each basin and assign every pixel to an elevation bin:

   | Bin | Elevation |
   |-----|-----------|
   | 1 | < 1500 m |
   | 2 | 1500–1800 m |
   | 3 | 1800–2500 m |
   | 4 | > 2500 m |

3. Calculate the area of each bin per basin.

**Inputs**
- `Data/DEM/*_dem.tif`: ASTER GDEM tiles
- Basin outlines (`shapes`) from `config.py`

**Outputs**
- `Data/DEM/<basin>_FixedElevBands.nc4`: clipped elevation and bin for each basin
- `Data/DEM/elev_bin_areas.csv`: area [m²] of each bin per basin, used to weight basin averages
- `Data/DEM/coarsened.nc4`: merged ASTER mosaic, coarsened to 15 arcsec

Needs ~3 GB of memory for the merge step.

In [ ]:
# Imports
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray  # noqa: F401  (registers the .rio accessor)
import xarray as xr
from rasterio.merge import merge as rio_merge

from config import data_folder, Basins, basin_names_list, shapes, elev_bins
from utils import calculate_area, visualize_basin_dem

from dask import delayed
import rasterio


dem_dir = os.path.join(data_folder, 'Data/DEM')
COARSEN = 15  # 1-arcsec ASTER pixels -> 15-arcsec
ELEV_EDGES = elev_bins # [0, 1500, 1800, 2500, 4315]
BIN_COLORS = ['#e41a1c', '#ff7f00', '#4daf4a', '#377eb8'] # colors for plotting elevation bins

NUM_WORKERS = 8  # set to the number of cores

In [ ]:
# Merge all ASTER tiles into one DEM and coarsen (needs ~3 GB at full resolution)
# dask parallelizes opening the tiles

tiles = glob.glob(os.path.join(dem_dir, '*_dem.tif'))
datasets = [delayed(rasterio.open)(fp) for fp in tiles]
arr, transform = delayed(rio_merge)(datasets).compute(num_workers=NUM_WORKERS)

ny, nx = arr.shape[1:]
x0, dx, _, y0, _, dy = transform.to_gdal()

elevation = xr.DataArray(
    arr[0], dims=('Latitude', 'Longitude'), name='Elevation',
    # pixel centers (the transform gives top-left corners)
    coords={'Latitude': y0 + (np.arange(ny) + 0.5) * dy,
            'Longitude': x0 + (np.arange(nx) + 0.5) * dx},
).coarsen(Latitude=COARSEN, Longitude=COARSEN, boundary='pad').mean()
del arr

dem_all = (xr.Dataset({'Elevation': elevation})
           .rio.write_crs('EPSG:4326')
           .rio.set_spatial_dims(x_dim='Longitude', y_dim='Latitude'))

In [ ]:
# %% Save the full study-area DEM (15-arcsec, not clipped to basins)
# Used by the storm-temperature notebook (low-elevation mask) and for figures
dem_all.to_netcdf(os.path.join(dem_dir, 'coarsened.nc4'))

In [ ]:
# Clip to each basin, assign elevation bins, save
elev_bin_areas = {}

for basin, name in zip(Basins, basin_names_list):
    polygon = shapes.loc[shapes['name'] == name, 'geometry'].iloc[0]
    dem = dem_all.rio.clip([polygon], crs='EPSG:4326')

    # Every finite elevation lands in 1–4; NaNs land in 5 and are masked
    bins = xr.apply_ufunc(np.digitize, dem['Elevation'], ELEV_EDGES)
    dem['ElevationBin'] = bins.astype('int16').where(bins < len(ELEV_EDGES))

    elev_bin_areas[basin] = calculate_area(dem, pixel_arcsec=COARSEN)  # adds 'Area' to dem
    print(f'{name}: areas [m²] {elev_bin_areas[basin]}')

    dem.drop_vars('Area').to_netcdf(os.path.join(dem_dir, f'{basin}_FixedElevBands.nc4'))
    visualize_basin_dem(dem, name)

In [ ]:
# Save bin areas (rows = ElevationBin 0–4, row 0 is a placeholder; columns = basins)
elev_bin_areas = pd.DataFrame(elev_bin_areas)
elev_bin_areas.to_csv(os.path.join(dem_dir, 'elev_bin_areas.csv'), index_label='ElevationBin')
elev_bin_areas